In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GEMINI_API_KEY']:
    print("GEMINI_API_KEY is set.")

GEMINI_API_KEY is set.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                            api_key=os.environ['GEMINI_API_KEY'])

## RAG IMPLEMENTATION WITH PDF

# 
Step -1 
Extracting Text from PDF


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("MicrosoftFabricOverview.pdf")
doc=loader.load()
doc

C:\Users\Dell\AppData\Local\Temp\ipykernel_23600\3629462828.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-11-14T18:35:13+01:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'title': 'PowerPoint Presentation', 'author': 'dmacpherson', 'moddate': '2025-11-14T18:35:13+01:00', 'source': 'MicrosoftFabricOverview.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Microsoft Fabric\nMicrosoft Fabric in a \nnutshell\nAndreas Rederer\nMunich, 14. November 2025'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2025-11-14T18:35:13+01:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_lab

In [5]:
for i in doc:
    i.metadata={"source":"MicrosoftFabricOverview.pdf",
                "author":"Microsoft"}
doc
    

[Document(metadata={'source': 'MicrosoftFabricOverview.pdf', 'author': 'Microsoft'}, page_content='Microsoft Fabric\nMicrosoft Fabric in a \nnutshell\nAndreas Rederer\nMunich, 14. November 2025'),
 Document(metadata={'source': 'MicrosoftFabricOverview.pdf', 'author': 'Microsoft'}, page_content='CIOs are accelerating \ntheir efforts to bring \nAI to their data estate\n“I am the Chief Information Officer \nand don’t want to be the Chief \nIntegration Officer. Help me translate \nAI to my competitive advantage.”\nEvery CIO, Every Enterprise\n2 Microsoft Fabric'),
 Document(metadata={'source': 'MicrosoftFabricOverview.pdf', 'author': 'Microsoft'}, page_content='The starting line: A complex, organically evolved data estate \n3\nBusiness team Business team Business team\nTech team\nEnterprise Data Sources\nMulti-Cloud | On-Premise | External\nIngestion\nStorage\nData transformation and BI\nData Management\nTech team\nIngestion\nStorage\nData transformation and BI\nData Management\nSeveral\nA

# 
Step-2 Chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks=splitter.split_documents(doc)
chunks
len(chunks)

70

#
Step-3 Embedding of chunks


In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
      
)

#
Step-4  Create Persisiting Vector DB

In [ ]:
from langchain_community.vectorstores import Chroma


Vectorstore = Chroma.from_documents(documents=chunks,
                                 embedding=embeddings,
                                 persist_directory="./VectorDB")


#
Step-5 Semantic Search

In [1]:
response=Vectorstore.similarity_search("Microsfoft Fabric?", k=3)
response[0].metadata
response[0].page_content


NameError: name 'Vectorstore' is not defined

#
Re Use Vector DB

In [6]:
from langchain_community.vectorstores import Chroma
vector_persist=Chroma(persist_directory="./VectorDB", embedding_function=embeddings)

C:\Users\Dell\AppData\Local\Temp\ipykernel_8024\2547923461.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_persist=Chroma(persist_directory="./VectorDB", embedding_function=embeddings)


#
Semantic Search using same Persisting Vector DB

In [7]:
response=vector_persist.similarity_search("wHAT IS Microsoft Fabric?", k=3)
response[0].metadata
response[0].page_content

'Databases\n Real-Time \nIntelligence\n Power BI\nData \nFactory\n Analytics\nMicrosoft Fabric\nThe unified data platform for AI transformation\nOneLake GovernanceAI\nFabric Platform'